# CmdStanPy → ArviZ Conversion Guide

This guide shows how to convert CmdStanPy outputs into ArviZ
`InferenceData` using `arviz-base`.


In [17]:
import arviz_base as azb
import cmdstanpy
from arviz_base.io_cmdstanpy import CmdStanPyConverter

In [18]:
import cmdstanpy
cmdstanpy.install_cmdstan()


CmdStan install directory: /Users/shivanipatel/.cmdstan
CmdStan version 2.38.0 already installed
Test model compilation


True

In [19]:
stan_code = """
data {
  int<lower=0> N;
  array[N] real y;
}
parameters {
  real mu;
  real<lower=0> sigma;
}
model {
  mu ~ normal(0, 5);
  sigma ~ exponential(1);

  y ~ normal(mu, sigma);
}
"""



In [20]:
with open("normal_model.stan", "w") as f:
    f.write(stan_code)



In [21]:
from pathlib import Path

stan_file = Path("normal_model.stan")

stan_file.write_text(stan_code)


179

In [22]:
from cmdstanpy import CmdStanModel
import numpy as np

model = CmdStanModel(stan_file="normal_model.stan")

data = {
    "N": 100,
    "y": np.random.normal(0, 1, size=100)
}

fit = model.sample(
    data=data,
    chains=2,
    iter_sampling=500,
    iter_warmup=500,
    seed=42,
)



13:00:42 - cmdstanpy - INFO - compiling stan file /Users/shivanipatel/arviz-base/docs/source/how_to/normal_model.stan to exe file /Users/shivanipatel/arviz-base/docs/source/how_to/normal_model
13:00:44 - cmdstanpy - INFO - compiled model executable: /Users/shivanipatel/arviz-base/docs/source/how_to/normal_model
13:00:44 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]

13:00:44 - cmdstanpy - INFO - CmdStan done processing.
13:00:44 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'normal_model.stan', line 14, column 2 to column 24)
Consider re-running with show_console=True if the above output is unclear!


In [23]:
fit.summary()


,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-35.820900,0.039389,0.965448,0.712926,-37.781900,-35.517800,-34.89650,609.284,620.917,101547.0,0.999612
mu,-0.067616,0.003073,0.088043,0.083609,-0.214207,-0.071669,0.08004,833.849,616.435,138975.0,1.000480
sigma,0.862569,0.001790,0.058328,0.055577,0.772873,0.858440,0.97115,1101.280,736.161,183546.0,0.998999


In [24]:
from arviz_base.io_cmdstanpy import CmdStanPyConverter
converter = CmdStanPyConverter(posterior=fit)

idata = converter.to_datatree()

idata

<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:  (chain: 2, draw: 500)
│       Coordinates:
│         * chain    (chain) int64 16B 0 1
│         * draw     (draw) int64 4kB 0 1 2 3 4 5 6 7 ... 493 494 495 496 497 498 499
│       Data variables:
│           mu       (chain, draw) float64 8kB -0.08332 -0.06989 ... -0.119 0.002904
│           sigma    (chain, draw) float64 8kB 0.7755 0.8227 0.883 ... 0.8431 0.8243
│       Attributes:
│           created_at:                 2026-02-05T07:30:44.683446+00:00
│           creation_library:           ArviZ
│           creation_library_version:   0.9.0.dev0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
└── Group: /sample_stats
        Dimensions:          (chain: 2, draw: 500)
        Coordinates:
          * chain            (chain) int64 16B 0 1
          * draw             (draw) int64 4kB 0 1 2 3 4 5 6 ... 494 495 496 497 498 499
        Data variables:
            lp               (chain, draw) float64 8kB -35.8 -34.98 ... -35.03 -35.36
            acceptance_rate  (chain, draw) float64 8kB 0.9282 1.0 ... 0.9738 0.9596
            step_size        (chain, draw) float64 8kB 1.005 1.005 1.005 ... 1.026 1.026
            tree_depth       (chain, draw) int64 8kB 2 1 2 2 2 1 1 2 ... 1 2 2 2 2 2 2 2
            n_steps          (chain, draw) int64 8kB 3 3 7 3 3 3 3 3 ... 3 7 3 3 7 3 3 7
            diverging        (chain, draw) bool 1kB False False False ... False False
            energy           (chain, draw) float64 8kB 36.85 35.6 35.37 ... 35.49 35.49
        Attributes:
            created_at:                 2026-02-05T07:30:44.685777+00:00
            creation_library:           ArviZ
            creation_library_version:   0.9.0.dev0
            creation_library_language:  Python
            inference_library:          cmdstanpy
            inference_library_version:  1.3.0

from arviz_base.io_cmdstanpy import CmdStanPyConverter

converter = CmdStanPyConverter(posterior=fit)

idata = converter.to_datatree()

idata


In [25]:
idata.posterior


<xarray.DataTree 'posterior'>
Group: /posterior
    Dimensions:  (chain: 2, draw: 500)
    Coordinates:
      * chain    (chain) int64 16B 0 1
      * draw     (draw) int64 4kB 0 1 2 3 4 5 6 7 ... 493 494 495 496 497 498 499
    Data variables:
        mu       (chain, draw) float64 8kB -0.08332 -0.06989 ... -0.119 0.002904
        sigma    (chain, draw) float64 8kB 0.7755 0.8227 0.883 ... 0.8431 0.8243
    Attributes:
        created_at:                 2026-02-05T07:30:44.683446+00:00
        creation_library:           ArviZ
        creation_library_version:   0.9.0.dev0
        creation_library_language:  Python
        inference_library:          cmdstanpy
        inference_library_version:  1.3.0

In [26]:
idata.sample_stats


<xarray.DataTree 'sample_stats'>
Group: /sample_stats
    Dimensions:          (chain: 2, draw: 500)
    Coordinates:
      * chain            (chain) int64 16B 0 1
      * draw             (draw) int64 4kB 0 1 2 3 4 5 6 ... 494 495 496 497 498 499
    Data variables:
        lp               (chain, draw) float64 8kB -35.8 -34.98 ... -35.03 -35.36
        acceptance_rate  (chain, draw) float64 8kB 0.9282 1.0 ... 0.9738 0.9596
        step_size        (chain, draw) float64 8kB 1.005 1.005 1.005 ... 1.026 1.026
        tree_depth       (chain, draw) int64 8kB 2 1 2 2 2 1 1 2 ... 1 2 2 2 2 2 2 2
        n_steps          (chain, draw) int64 8kB 3 3 7 3 3 3 3 3 ... 3 7 3 3 7 3 3 7
        diverging        (chain, draw) bool 1kB False False False ... False False
        energy           (chain, draw) float64 8kB 36.85 35.6 35.37 ... 35.49 35.49
    Attributes:
        created_at:                 2026-02-05T07:30:44.685777+00:00
        creation_library:           ArviZ
        creation_library_version:   0.9.0.dev0
        creation_library_language:  Python
        inference_library:          cmdstanpy
        inference_library_version:  1.3.0


## Convert CmdStanPy output to ArviZ DataTree

We use CmdStanPyConverter to convert CmdStanPy sampling results
into an ArviZ-compatible DataTree structure.

This allows analysis and visualization using ArviZ tools.
